# Week 5 - Day 4: Structured Output — Claude JSON Mode
**Goal:** Force Claude to return validated JSON, not free text.

In production, you can't have Claude sometimes return paragraphs and sometimes
return JSON. You need **guaranteed structure** every time. Today you learn how.

In [12]:
import os
os.environ["ANTHROPIC_API_KEY"] = "your-key-here"   # REMOVE before pushing to GitHub!"


In [13]:
import anthropic
import json

client = anthropic.Anthropic()
MODEL = "claude-sonnet-5"

def get_text(message):
    """Extract text from Claude response (handles thinking blocks)."""
    return next(block.text for block in message.content if block.type == "text")

print("Ready!")

Ready!


---
## Part 1: Basic JSON Prompt

New concepts:
- Tell Claude to respond in JSON format in the **system prompt**
- Use `json.loads(text)` to parse the JSON string into a Python dictionary
- `json.loads()` is the reverse of `json.dumps()` — string → dict

In [14]:
# TODO: Ask Claude to analyze a stock and return JSON
#
#   text = "DBS Group reported Q3 2024 earnings of $2.63 billion, up 15% year-over-year."
#
#   message = client.messages.create(
#       model=MODEL,
#       max_tokens=500,
#       system="You are a financial analyst. Always respond in valid JSON format only. No other text.",
#       messages=[{"role": "user", "content": f"Analyze this and return JSON with keys: company, metric, value, change, sentiment.\n\n{text}"}]
#   )
#
#   raw = get_text(message)
#   print("Raw response:")
#   print(raw)

In [15]:
text = "DBS Group reported Q3 2024 earnings of $2.63 billion, up 15% year-over-year."

In [16]:
#   message = client.messages.create(
#       model=MODEL,
#       max_tokens=500,
#       system="You are a financial analyst. Always respond in valid JSON format only. No other text.",
#       messages=[{"role": "user", "content": f"Analyze this and return JSON with keys: company, metric, value, change, sentiment.\n\n{text}"}]
#   )

In [22]:
message = client.messages.create(
    model = MODEL,
    max_tokens = 500,
    system = "You are a financial analyst. Always respond in valid JSON format only. No other text.",
    messages=[{"role": "user", "content": f"Analyze this and return JSON with keys: company, metric, value, change, sentiment.\n\n{text}"}]
)

In [23]:
raw = get_text(message)
print("Raw response:")
print(raw)

Raw response:
```json
{
  "company": "DBS Group",
  "metric": "Q3 2024 earnings",
  "value": "$2.63 billion",
  "change": "+15% year-over-year",
  "sentiment": "positive"
}
```


In [40]:
data = parse_json_response(raw)
print(f"\nParsed dict:")
print(f"  Company: {data['company']}")
print(f"  Metric:  {data['metric']}")
print(f"  Value:   {data['value']}")
print(f"  Change:  {data['change']}")
print(f"  Sentiment: {data['sentiment']}")


Parsed dict:
  Company: DBS Group
  Metric:  Q3 2024 earnings
  Value:   $2.63 billion
  Change:  +15% year-over-year
  Sentiment: positive


In [41]:
data

{'company': 'DBS Group',
 'metric': 'Q3 2024 earnings',
 'value': '$2.63 billion',
 'change': '+15% year-over-year',
 'sentiment': 'positive'}

In [ ]:
# TODO: Parse the JSON string into a Python dict
#   data = json.loads(raw)
#   print(f"\nParsed dict:")
#   print(f"  Company: {data['company']}")
#   print(f"  Metric:  {data['metric']}")
#   print(f"  Value:   {data['value']}")
#   print(f"  Change:  {data['change']}")
#   print(f"  Sentiment: {data['sentiment']}")

---
## Part 2: Handle Markdown Wrapping

New concept:
- Claude often wraps JSON in \`\`\`json ... \`\`\` markdown blocks
- You need to strip these before parsing
- `text.strip()` removes whitespace, `.removeprefix()` / `.removesuffix()` remove specific strings

In [28]:
def parse_json_response(raw_text):
    """Clean markdown wrapping and parse JSON."""
    cleaned = raw_text.strip()
    cleaned = cleaned.removeprefix("```json").removeprefix("```")
    cleaned = cleaned.removesuffix("```")
    cleaned = cleaned.strip()
    return json.loads(cleaned)

In [31]:
cleaned1 = raw.strip()
cleaned1

'```json\n{\n  "company": "DBS Group",\n  "metric": "Q3 2024 earnings",\n  "value": "$2.63 billion",\n  "change": "+15% year-over-year",\n  "sentiment": "positive"\n}\n```'

In [34]:
cleaned1 = cleaned1.removeprefix("```json").removeprefix("```")
cleaned1

'\n{\n  "company": "DBS Group",\n  "metric": "Q3 2024 earnings",\n  "value": "$2.63 billion",\n  "change": "+15% year-over-year",\n  "sentiment": "positive"\n}\n```'

In [36]:
cleaned1 = cleaned1.removesuffix("```")
cleaned1

'\n{\n  "company": "DBS Group",\n  "metric": "Q3 2024 earnings",\n  "value": "$2.63 billion",\n  "change": "+15% year-over-year",\n  "sentiment": "positive"\n}\n'

In [37]:
cleaned1 = cleaned1.strip()
cleaned1

'{\n  "company": "DBS Group",\n  "metric": "Q3 2024 earnings",\n  "value": "$2.63 billion",\n  "change": "+15% year-over-year",\n  "sentiment": "positive"\n}'

In [ ]:
Raw response:
```json
{
  "company": "DBS Group",
  "metric": "Q3 2024 earnings",
  "value": "$2.63 billion",
  "change": "+15% year-over-year",
  "sentiment": "positive"
}
```

In [ ]:
# TODO: Build a function to clean and parse Claude's JSON response
#
#   def parse_json_response(raw_text):
#       """Clean markdown wrapping and parse JSON."""
#       cleaned = raw_text.strip()
#       cleaned = cleaned.removeprefix("```json").removeprefix("```")
#       cleaned = cleaned.removesuffix("```")
#       cleaned = cleaned.strip()
#       return json.loads(cleaned)
#
#   # Test it
#   test_raw = '```json\n{"company": "DBS", "price": 41.20}\n```'
#   result = parse_json_response(test_raw)
#   print(f"Parsed: {result}")
#   print(f"Company: {result['company']}")

In [38]:
def parse_json_response(raw_text):
    """Clean markdown wrapping and parse JSON."""
    cleaned = raw_text.strip()
    cleaned = cleaned.removeprefix("```json").removeprefix("```")
    cleaned = cleaned.removesuffix("```")
    cleaned = cleaned.strip()
    return json.loads(cleaned)

In [39]:
test_raw = '```json\n{"company": "DBS", "price": 41.20}\n```'
result = parse_json_response(test_raw)
print(f"Parsed: {result}")
print(f"Company: {result['company']}")

Parsed: {'company': 'DBS', 'price': 41.2}
Company: DBS


---
## Part 3: Validate JSON Structure

New concepts:
- Even if JSON parses, it might be missing keys you need
- Check that all required keys exist before using the data
- Return `(True, data)` or `(False, error_message)` — same tuple pattern from Week 4

In [ ]:
# TODO: Build a validator function
#
#   def validate_analysis(data, required_keys):
#       """Check that all required keys exist in the parsed JSON."""
#       missing = [key for key in required_keys if key not in data]
#       if missing:
#           return (False, f"Missing keys: {missing}")
#       return (True, data)
#
#   # Test with good data
#   good = {"company": "DBS", "metric": "earnings", "value": "$2.63B", "change": "+15%", "sentiment": "positive"}
#   valid, result = validate_analysis(good, ["company", "metric", "value", "change", "sentiment"])
#   print(f"Good data — valid: {valid}")
#
#   # Test with bad data (missing keys)
#   bad = {"company": "DBS", "metric": "earnings"}
#   valid, result = validate_analysis(bad, ["company", "metric", "value", "change", "sentiment"])
#   print(f"Bad data — valid: {valid}, reason: {result}")

In [42]:
def validate_analysis(data, required_keys):
    """Check that all required keys exist in the parsed JSON."""
    missing = [key for key in required_keys if key not in data]
    if missing:
        return (False, f"Missing keys: {missing}")
    return (True, data)

In [43]:
good = {"company": "DBS", "metric": "earnings", "value": "$2.63B", "change": "+15%", "sentiment": "positive"}
valid, result = validate_analysis(good, ["company", "metric", "value", "change", "sentiment"])
print(f"Good data — valid: {valid}")

Good data — valid: True


In [44]:
bad = {"company": "DBS", "metric": "earnings"}
valid, result = validate_analysis(bad, ["company", "metric", "value", "change", "sentiment"])
print(f"Bad data — valid: {valid}, reason: {result}")

Bad data — valid: False, reason: Missing keys: ['value', 'change', 'sentiment']


---
## Part 4: Safe Analysis Function

Combine everything: call Claude → clean response → parse JSON → validate.
Handle errors at every step so it never crashes.

In [ ]:
# TODO: Build the complete safe analysis function
#
#   REQUIRED_KEYS = ["company", "sector", "metric", "value", "sentiment", "confidence"]
#
#   def analyze_stock_news(text):
#       """Send text to Claude, get validated JSON back."""
#       try:
#           message = client.messages.create(
#               model=MODEL,
#               max_tokens=500,
#               system="""You are a financial analyst. Always respond in valid JSON only.
#   Return exactly these keys:
#   - company: company name
#   - sector: industry sector
#   - metric: what financial metric is mentioned
#   - value: the numeric value
#   - sentiment: "positive", "negative", or "neutral"
#   - confidence: "high", "medium", or "low"
#   No other text, just the JSON object.""",
#               messages=[{"role": "user", "content": text}]
#           )
#           raw = get_text(message)
#           data = parse_json_response(raw)
#           valid, result = validate_analysis(data, REQUIRED_KEYS)
#           if not valid:
#               return {"error": result}
#           return data
#       except json.JSONDecodeError as e:
#           return {"error": f"JSON parse failed: {e}"}
#       except Exception as e:
#           return {"error": f"API call failed: {e}"}
#
#   print("Function ready!")

In [47]:
REQUIRED_KEYS = ["company", "sector", "metric", "value", "sentiment", "confidence"]

def analyze_stock_news(text):
    """Send text to Claude, get validated JSON back."""
    try:
        message = client.messages.create(
            model=MODEL,
            max_tokens=500,
            system="""You are a financial analyst. Always respond in valid JSON only.
Return exactly these keys:
- company: company name
- sector: industry sector
- metric: what financial metric is mentioned
- value: the numeric value
- sentiment: "positive", "negative", or "neutral"
- confidence: "high", "medium", or "low"
No other text, just the JSON object.""",
            messages=[{"role": "user", "content": text}]
        )
        raw = get_text(message)
        data = parse_json_response(raw)
        valid, result = validate_analysis(data, REQUIRED_KEYS)
        if not valid:
            return {"error": result}
        return data
    except json.JSONDecodeError as e:
        return {"error": f"JSON parse failed: {e}"}
    except Exception as e:
        return {"error": f"API call failed: {e}"}

print("Function ready!")

Function ready!


---
## Part 5: Batch Analysis with Validation

Process multiple headlines and collect structured results.

In [ ]:
# TODO: Define test headlines
#
#   headlines = [
#       "DBS Group reported record Q3 earnings of $2.63 billion, up 15% YoY.",
#       "Singapore Airlines passenger load fell to 82% amid weaker travel demand.",
#       "CapitaLand Ascendas REIT acquired a logistics property in Changi for $180M.",
#       "Sea Limited's revenue grew 23% to $3.3B, beating analyst expectations."
#   ]
#
#   print(f"Processing {len(headlines)} headlines...")

In [48]:
headlines = [
      "DBS Group reported record Q3 earnings of $2.63 billion, up 15% YoY.",
      "Singapore Airlines passenger load fell to 82% amid weaker travel demand.",
      "CapitaLand Ascendas REIT acquired a logistics property in Changi for $180M.",
      "Sea Limited's revenue grew 23% to $3.3B, beating analyst expectations."
  ]

print(f"Processing {len(headlines)} headlines...")

Processing 4 headlines...


In [ ]:
# TODO: Process all headlines and collect results
#
#   results = []
#   for i, headline in enumerate(headlines):
#       print(f"\n[{i+1}/{len(headlines)}] {headline[:50]}...")
#       data = analyze_stock_news(headline)
#       if "error" in data:
#           print(f"  ERROR: {data['error']}")
#       else:
#           print(f"  Company: {data['company']}")
#           print(f"  Sentiment: {data['sentiment']} (confidence: {data['confidence']})")
#       results.append(data)
#
#   print(f"\nDone! {len(results)} results collected.")vb

In [49]:
results = []
for i, headline in enumerate(headlines):
  print(f"\n[{i+1}/{len(headlines)}] {headline[:50]}...")
  data = analyze_stock_news(headline)
  if "error" in data:
      print(f"  ERROR: {data['error']}")
  else:
      print(f"  Company: {data['company']}")
      print(f"  Sentiment: {data['sentiment']} (confidence: {data['confidence']})")
  results.append(data)

print(f"\nDone! {len(results)} results collected.")


[1/4] DBS Group reported record Q3 earnings of $2.63 bil...
  Company: DBS Group
  Sentiment: positive (confidence: high)

[2/4] Singapore Airlines passenger load fell to 82% amid...
  Company: Singapore Airlines
  Sentiment: negative (confidence: high)

[3/4] CapitaLand Ascendas REIT acquired a logistics prop...
  Company: CapitaLand Ascendas REIT
  Sentiment: positive (confidence: high)

[4/4] Sea Limited's revenue grew 23% to $3.3B, beating a...
  Company: Sea Limited
  Sentiment: positive (confidence: high)

Done! 4 results collected.


In [50]:
results

[{'company': 'DBS Group',
  'sector': 'Banking/Financial Services',
  'metric': 'Q3 earnings',
  'value': '2.63 billion',
  'sentiment': 'positive',
  'confidence': 'high'},
 {'company': 'Singapore Airlines',
  'sector': 'Airlines',
  'metric': 'Passenger Load Factor',
  'value': 82,
  'sentiment': 'negative',
  'confidence': 'high'},
 {'company': 'CapitaLand Ascendas REIT',
  'sector': 'Real Estate (REIT)',
  'metric': 'Acquisition Value',
  'value': 180000000,
  'sentiment': 'positive',
  'confidence': 'high'},
 {'company': 'Sea Limited',
  'sector': 'Technology',
  'metric': 'Revenue',
  'value': '3.3B (23% growth)',
  'sentiment': 'positive',
  'confidence': 'high'}]

---
## Part 6: Results to DataFrame

New concept:
- `pd.DataFrame(list_of_dicts)` — convert a list of dictionaries into a DataFrame
- Each dict becomes a row, keys become column names

In [ ]:
# TODO: Convert results to a DataFrame for analysis
#
#   import pandas as pd
#
#   # Filter out any errors
#   valid_results = [r for r in results if "error" not in r]
#   print(f"Valid results: {len(valid_results)} / {len(results)}")
#
#   df = pd.DataFrame(valid_results)
#   df

In [51]:
import pandas as pd

# Filter out any errors
valid_results = [r for r in results if "error" not in r]
print(f"Valid results: {len(valid_results)} / {len(results)}")

df = pd.DataFrame(valid_results)
df

Valid results: 4 / 4


,company,sector,metric,value,sentiment,confidence
0,DBS Group,Banking/Financial Services,Q3 earnings,2.63 billion,positive,high
1,Singapore Airlines,Airlines,Passenger Load Factor,82,negative,high
2,CapitaLand Ascendas REIT,Real Estate (REIT),Acquisition Value,180000000,positive,high
3,Sea Limited,Technology,Revenue,3.3B (23% growth),positive,high


In [ ]:
# TODO: Analyze the structured results
#   print("Sentiment breakdown:")
#   print(df["sentiment"].value_counts())
#   print()
#   print("Confidence breakdown:")
#   print(df["confidence"].value_counts())

In [52]:
print("Sentiment breakdown:")
print(df["sentiment"].value_counts())
print()
print("Confidence breakdown:")
print(df["confidence"].value_counts())

Sentiment breakdown:
sentiment
positive    3
negative    1
Name: count, dtype: int64

Confidence breakdown:
confidence
high    4
Name: count, dtype: int64
